# Group 12 — PLSA All-20 Topic-Count Study

This is a new model-order experiment on all 20 categories of 20 Newsgroups. It compares PLSA with K = 2, 5, 8, 10, and 20 while holding the corpus, split, preprocessing, vocabulary, queries, retrieval metrics, and baselines fixed.

The split is 60% training, 20% validation, and 20% test. Validation results select a K for each PLSA scoring method. Test results for every predeclared K are reported as a model-order comparison; do not use those test scores to retune the model. Category labels are used for splitting and evaluation only, not as PLSA training targets.


All shared implementations are imported from `src/`. Run this notebook from the repository root. Old execution outputs were cleared during refactoring. The final cell writes tables and metadata for `Result.py`; experiment-specific loops remain here.

## 0. Install dependencies

Run once in a notebook environment if required. In Colab, use the first line in a code cell.

In [ ]:
# Install from the repository root: pip install -r requirements.txt
# Then: python -m spacy download en_core_web_sm


## 1. Imports and reproducibility

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

SEED = 42
rng = np.random.default_rng(SEED)
np.random.seed(SEED)

from src.config import EXPERIMENTS, SUPERGROUPS, CATEGORY_QUERIES, SUPERGROUP_QUERIES, QUERY_BANK
from src.dataset import load_corpus
from src.preprocessing import preprocess_texts, split_train_test, split_train_val_test, vectorize_counts
from src.plsa_model import PLSA, train_plsa_restarts
from src.retrieval import fit_baselines, plsa_query_log_likelihood, background_word_probabilities
from src.metrics import metrics_at_k
from src.experiments import make_mixed_document, average_rank_score
from src.results import export_results
EXPERIMENT = 'All20_K_Study'
CONFIG = EXPERIMENTS[EXPERIMENT]


## 2. Load the full 20-category corpus

Labels define splits and relevance only; PLSA trains on counts without labels.

In [ ]:
# Shared loader; corpus selection remains an explicit experimental choice.
corpus = load_corpus(CONFIG["corpus"], seed=SEED)
documents, labels, target_names = corpus.documents, corpus.labels, corpus.target_names
original_labels = corpus.original_labels
source_category_names = corpus.source_category_names
source_to_supergroup = {category: group for group, members in SUPERGROUPS.items() for category in members}
print("Corpus:", CONFIG["corpus"], "Documents:", len(documents))
for i, label in enumerate(target_names):
    print(label, int((labels == i).sum()))


> Historical interpretation from before the shared-code refactor; rerun to verify any numerical claims below.

### Main corpus versus boundary experiment

The model and retrieval evaluation use the **full 20-category corpus**. The controlled boundary experiment still uses two chosen source categories to create interpretable 90:10 through 10:90 word mixtures. This does not mean the PLSA model was trained on only those two categories.

## 3. Train/test split and preprocessing

Split the raw documents first. Fit the vocabulary only on the training split to avoid vocabulary leakage from the test documents.

In [ ]:
# Split raw text first. Fit vocabulary only on training text.
docs_train, docs_val, docs_test, y_train, y_val, y_test = split_train_val_test(documents, labels, seed=SEED)
docs_train_processed = preprocess_texts(docs_train)
docs_val_processed = preprocess_texts(docs_val)
docs_test_processed = preprocess_texts(docs_test)
count_vectorizer, X_train, X_val, X_test = vectorize_counts(docs_train_processed, docs_val_processed, docs_test_processed)
vocabulary = count_vectorizer.get_feature_names_out()
nonempty_test_mask = np.asarray([bool(doc.strip()) for doc in docs_test_processed])
print("Train/test count matrices:", X_train.shape, X_test.shape)
print("Non-empty test texts:", int(nonempty_test_mask.sum()))


## 4. PLSA theory

For document \(d\), word \(w\), and latent topic \(z\):

\[
P(w\mid d)=\sum_{z=1}^{K}P(w\mid z)P(z\mid d)
\]

- \(\theta_{d,z}=P(z\mid d)\): document-topic distribution.
- \(\phi_{z,w}=P(w\mid z)\): topic-word distribution.
- Each row of \(\theta\) and \(\phi\) sums to one.

For every observed document-word pair, the E-step computes the posterior responsibility:

\[
P(z\mid d,w)=\frac{P(z\mid d)P(w\mid z)}
{\sum_{z'}P(z'\mid d)P(w\mid z')}
\]

The M-step uses the expected word counts to update both distributions. The process repeats until the log-likelihood stops improving.

In [ ]:
# The single EM and fold-in implementation lives in src/plsa_model.py.
from src.plsa_model import PLSA


## 5. Multiple random restarts

EM can settle at different local optima. Fit several initializations and select the model with the largest training log-likelihood. This selection criterion is only about training fit; final retrieval quality must still be evaluated on held-out documents.

In [ ]:
# train_plsa_restarts is imported from src above.

K = 5
best_plsa, restart_runs = train_plsa_restarts(
    X_train, n_topics=K, n_restarts=5, max_iter=100
)
theta_train = best_plsa.transform()
theta_test = best_plsa.fold_in(X_test, max_iter=100)
print("theta_train:", theta_train.shape)
print("theta_test:", theta_test.shape)


## 6. Inspect topics and convergence

Topic IDs are arbitrary: topic 0 is not inherently sports or politics. Interpret each topic by its highest-probability words, and use category-level averages only as a diagnostic.

In [ ]:
for topic_id, words in best_plsa.top_words(vocabulary, n=12).items():
    print(f"\nTopic {topic_id}:")
    print(", ".join(f"{word} ({prob:.3f})" for word, prob in words))

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(best_plsa.log_likelihoods)+1),
         best_plsa.log_likelihoods, marker="o", markersize=3)
plt.xlabel("EM iteration")
plt.ylabel("Training log-likelihood")
plt.title("PLSA convergence")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 7. Controlled topic-boundary experiment

Generate artificial documents by sampling word tokens from source documents in known categories. We then infer topic mixtures with folding-in and compare the known source proportions with the estimated probabilities.

**Interpretation caution:** a source-word ratio is not guaranteed to equal a latent-topic probability. Topics can split a category or overlap categories. We therefore identify sports-associated and politics-associated topics from training-set category averages, and interpret the result as an experiment rather than ground truth equivalence.

In [ ]:
sports_class = target_names.index("rec.sport.baseball")
politics_class = target_names.index("talk.politics.misc")

class_topic_means = np.vstack([
    theta_train[y_train == c].mean(axis=0)
    for c in range(len(target_names))
])
sports_topic = int(class_topic_means[sports_class].argmax())
politics_topic = int(class_topic_means[politics_class].argmax())

print("Sports-associated topic:", sports_topic)
print("Politics-associated topic:", politics_topic)
if sports_topic == politics_topic:
    print("Warning: both categories map to the same dominant topic; interpret estimates cautiously.")


# make_mixed_document is imported from src above.

boundary_rng = np.random.default_rng(SEED + 77)
ratios = [0.9, 0.7, 0.5, 0.3, 0.1]
boundary_rows = []

for ratio in ratios:
    for replicate in range(20):
        mixed_X = make_mixed_document(
            X_train, y_train, sports_class, politics_class,
            ratio, boundary_rng, n_tokens=300
        )
        estimate = best_plsa.fold_in(mixed_X, max_iter=100)[0]
        boundary_rows.append({
            "source_category_a": "rec.sport.baseball",
            "source_category_b": "talk.politics.misc",
            "actual_sports_word_ratio": ratio,
            "actual_politics_word_ratio": 1-ratio,
            "estimated_sports_topic": estimate[sports_topic],
            "estimated_politics_topic": estimate[politics_topic],
            "replicate": replicate
        })

boundary_df = pd.DataFrame(boundary_rows)
boundary_summary = boundary_df.groupby("actual_sports_word_ratio").agg(
    estimated_sports_mean=("estimated_sports_topic", "mean"),
    estimated_sports_std=("estimated_sports_topic", "std"),
    estimated_politics_mean=("estimated_politics_topic", "mean"),
    estimated_politics_std=("estimated_politics_topic", "std")
).reset_index()
display(boundary_summary)

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot([0, 1], [0, 1], "k--", label="Ideal reference")
plt.errorbar(
    boundary_summary["actual_sports_word_ratio"],
    boundary_summary["estimated_sports_mean"],
    yerr=boundary_summary["estimated_sports_std"],
    marker="o", capsize=3, label="Estimated sports-topic probability"
)
plt.xlabel("Actual sports word-token proportion")
plt.ylabel("Estimated sports-associated topic probability")
plt.title("PLSA topic-boundary experiment")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 8. TF-IDF/VSM and LSI baselines

All baselines are fitted on training documents only. LSI uses truncated SVD on TF-IDF representations; it is a geometric low-rank representation, unlike PLSA's normalized probability distributions.

In [ ]:
# All baselines use the shared train-only fitting implementation.
tfidf_vectorizer, lsi_model, tfidf_splits, lsi_splits = fit_baselines(
    docs_train_processed, docs_val_processed, docs_test_processed, seed=SEED
)
X_train_tfidf, X_val_tfidf, X_test_tfidf = tfidf_splits
lsi_train, lsi_val, lsi_test = lsi_splits



## 9. Retrieval and model-order comparison

We reuse the same 20 fixed category queries from the all-20 baseline. PLSA is evaluated with cosine similarity between folded-in topic mixtures and with query likelihood. For each K, the best of three random restarts is chosen by training log-likelihood. Validation MAP@10 selects K separately for each PLSA scoring method. The final test table reports all five predeclared K values alongside TF-IDF and LSI; test results are for comparison and are not used to tune K.


In [ ]:
category_queries = {name: CATEGORY_QUERIES[name] for name in target_names}
assert set(category_queries) == set(target_names)

K_GRID = [2, 5, 8, 10, 20]
N_RESTARTS = 3
TOP_K = 10

# metrics_at_k is imported from src above.

def evaluate_model(model, theta_docs, split_name, labels, processed_docs,
                   tfidf_docs, lsi_docs, k_topics):
    eligible = np.asarray([bool(text.strip()) for text in processed_docs])
    y_eval = np.asarray(labels)[eligible]
    theta_eval = theta_docs[eligible]
    tfidf_eval = tfidf_docs[eligible]
    lsi_eval = lsi_docs[eligible]
    rows = []
    for category, query_text in category_queries.items():
        target = target_names.index(category)
        processed_query = preprocess_texts([query_text])[0]
        q_counts = count_vectorizer.transform([processed_query]).astype(np.float64)
        q_tfidf = tfidf_vectorizer.transform([processed_query])
        q_lsi = lsi_model.transform(q_tfidf)
        q_theta = model.fold_in(q_counts, max_iter=100)
        scores = {
            "TF-IDF/VSM": cosine_similarity(q_tfidf, tfidf_eval).ravel(),
            "LSI": cosine_similarity(q_lsi, lsi_eval).ravel(),
            "PLSA cosine": cosine_similarity(q_theta, theta_eval).ravel(),
        }
        scores["PLSA query likelihood"] = plsa_query_log_likelihood(
            q_counts, theta_eval, model.phi_, smoothing=1e-12
        )
        for method, method_scores in scores.items():
            rows.append({"split": split_name, "K": k_topics,
                         "query_category": category, "query_text": query_text,
                         "method": method,
                         **metrics_at_k(method_scores, y_eval, target)})
    return rows

models_by_k = {}
model_order_rows = []
validation_rows = []
test_rows = []
for k_topics in K_GRID:
    print(f"\nFitting K={k_topics} with {N_RESTARTS} restarts")
    model, runs = train_plsa_restarts(
        X_train, n_topics=k_topics, n_restarts=N_RESTARTS,
        max_iter=80, tol=1e-4, base_seed=SEED + 100 * k_topics
    )
    models_by_k[k_topics] = model
    model_order_rows.append({
        "K": k_topics, "iterations": model.n_iter_,
        "train_log_likelihood": model.log_likelihoods[-1],
        "restart_count": len(runs),
    })
    theta_val_k = model.fold_in(X_val, max_iter=100)
    theta_test_k = model.fold_in(X_test, max_iter=100)
    validation_rows.extend(evaluate_model(
        model, theta_val_k, "validation", y_val, docs_val_processed,
        X_val_tfidf, lsi_val, k_topics
    ))
    test_rows.extend(evaluate_model(
        model, theta_test_k, "test", y_test, docs_test_processed,
        X_test_tfidf, lsi_test, k_topics
    ))

model_order_df = pd.DataFrame(model_order_rows)
validation_df = pd.DataFrame(validation_rows)
test_all_k_df = pd.DataFrame(test_rows)

validation_summary = validation_df.groupby(["K", "method"])[
    ["Precision@10", "Recall@10", "F1@10", "AP@10", "MRR"]
].mean().reset_index()
test_plsa_df = test_all_k_df[test_all_k_df["method"].str.startswith("PLSA")]
test_baselines_df = test_all_k_df[
    (test_all_k_df["K"] == K_GRID[0]) &
    (test_all_k_df["method"].isin(["TF-IDF/VSM", "LSI"]))
]
test_summary = test_plsa_df.groupby(["K", "method"])[
    ["Precision@10", "Recall@10", "F1@10", "AP@10", "MRR"]
].mean().reset_index()
baseline_summary = test_baselines_df.groupby("method")[
    ["Precision@10", "Recall@10", "F1@10", "AP@10", "MRR"]
].mean()

val_ll = validation_summary[validation_summary["method"] == "PLSA query likelihood"].sort_values(
    ["AP@10", "Precision@10"], ascending=False
)
val_cos = validation_summary[validation_summary["method"] == "PLSA cosine"].sort_values(
    ["AP@10", "Precision@10"], ascending=False
)
best_k_ll = int(val_ll.iloc[0]["K"])
best_k_cos = int(val_cos.iloc[0]["K"])

print("\nTraining fit and convergence:")
display(model_order_df)
print("\nValidation retrieval by K (use these to choose K):")
display(validation_summary.sort_values(["method", "K"]))
print(f"Validation-selected K for PLSA query likelihood: {best_k_ll}")
print(f"Validation-selected K for PLSA cosine: {best_k_cos}")
print("\nFinal held-out test baseline results:")
display(baseline_summary)
print("\nFinal held-out PLSA test results for every predeclared K:")
display(test_summary.sort_values(["method", "K"]))


In [ ]:

test_plsa_plot = test_summary.pivot(index="K", columns="method", values="AP@10")
test_plsa_plot.plot(marker="o", figsize=(9, 5))
plt.axhline(baseline_summary.loc["TF-IDF/VSM", "AP@10"], linestyle="--", label="TF-IDF/VSM")
plt.axhline(baseline_summary.loc["LSI", "AP@10"], linestyle=":", label="LSI")
plt.title("All-20 held-out test MAP@10 by PLSA topic count")
plt.xlabel("PLSA topic count K")
plt.ylabel("Macro-averaged AP@10")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


## 10. Model-order interpretation

The preceding sweep compares K = 2, 5, 8, 10, and 20 on identical data and queries. Use validation MAP@10 to select K; interpret test scores as a predeclared sensitivity comparison, not as a second opportunity to tune. Training log-likelihood is included as a fit/convergence diagnostic and is not itself evidence of better retrieval or generalization.


In [ ]:

plsa_by_category = test_plsa_df.pivot_table(
    index=["K", "query_category"], columns="method", values="AP@10"
).reset_index()
display(plsa_by_category)



## 11. Optional extension: tempered E-step (exploratory)

Hofmann's paper introduces Tempered EM as an entropy-regularized fitting procedure and recommends evaluating held-out performance while changing the inverse temperature. The `beta` option in the class is a simplified tempered E-step, **not a full reproduction of the paper's TEM schedule**. Treat this section as exploratory only, and don't claim that it fully implements TEM.

For a proper TEM experiment, start with standard EM at beta=1, lower beta in small steps, run tempered iterations, and monitor held-out predictive likelihood/perplexity; stop when held-out performance deteriorates. A rigorous implementation should define and document the exact schedule and stopping rule.

In [ ]:
tempered_rows = []
for beta in [1.0, 0.9, 0.8, 0.7]:
    model = PLSA(
        n_topics=K, max_iter=80, tol=1e-4,
        random_state=SEED, beta=beta
    ).fit(X_train)
    tempered_rows.append({
        "beta": beta,
        "iterations": model.n_iter_,
        "train_log_likelihood": model.log_likelihoods[-1],
        "warning": "Simplified tempered E-step; not full TEM"
    })

tempered_df = pd.DataFrame(tempered_rows)
display(tempered_df)


## 12. Keep results in the notebook

Model-order, validation, and held-out test tables are displayed inline. No experiment CSV files are written.


## 13. Limitations to state in the report

1. PLSA is trained on all 20 categories, but learned topics do not necessarily correspond one-to-one to the 20 human labels.
2. Synthetic source-word proportions in the baseball/politics boundary experiment are not identical to latent-topic probabilities.
3. Selecting the best training likelihood does not guarantee the best retrieval performance.
4. Category labels are only a proxy for relevance; manually judged query-document relevance would be stronger.
5. The 20 representative queries are manually authored, and query wording can affect ranking. Include the query set in the report.
6. Hofmann's paper used MED, CRAN, CACM and CISI collections; this project uses 20 Newsgroups and does not reproduce the paper's exact numerical results.
7. The optional beta experiment is a simplified tempered E-step unless the full TEM schedule and held-out stopping criterion are implemented.
8. A capped vocabulary and sparse count matrix are used to keep computation manageable.


## 14. Reproducibility and interpretation checks

Before interpreting the comparison, verify:

1. **Preprocessing parity:** PLSA, TF-IDF, LSI, and query text all use `preprocess_texts`; vocabulary fitting uses training documents only.
2. **Empty documents/queries:** report the number of empty documents after preprocessing. Empty test documents cannot provide meaningful evidence for any retrieval method; report their count and consider excluding them consistently from retrieval evaluation (while retaining original labels/index mapping).
3. **PLSA topic interpretation:** topic IDs are arbitrary. Identify sports/politics topics by average training-document topic probabilities, then inspect top words and example documents before naming them.
4. **Boundary experiment:** the input ratio is the proportion of sampled tokens sourced from each category, not a guarantee that every token is category-exclusive. Shared vocabulary and imperfect learned topics explain deviations from the ideal line.
5. **Model order:** training log-likelihood generally improves with larger K; do not select K from training likelihood alone. Consider topic coherence/interpretability and held-out likelihood or retrieval results.
6. **No claims from stale outputs:** after changing preprocessing or vectorizer settings, rerun all cells from the top so every table and plot reflects the same code state.


## Export this run

Save result tables, convergence, topic words, settings and source hashes. `Result.py` reads these files without repeating model training.

In [ ]:
# This is reached only after all experiment cells complete successfully.
result_directory = export_results(EXPERIMENT, globals())
print("Results saved to:", result_directory)
